# Verify cameras

This notebook inventories connected cameras, tests frame capture, and shows labelled live previews so that each physical camera can be matched to its terminal. Press **q** in a preview window to exit it.

In [1]:
import sys
import re
import socket
from pathlib import Path
from pprint import pprint

sys.path.append("..")

import cv2
from exp_run_config import Config
Config.PROJECTNAME = 'BerryPicker'
print(f'OpenCV version: {cv2.__version__}')

OpenCV version: 5.0.0


In [2]:
def video_index(device_path):
    match = re.search(r'video(\d+)$', str(device_path))
    return int(match.group(1)) if match else None

# Stable names identify the USB terminal more reliably than probing arbitrary indices.
stable_paths = []
for directory in (Path('/dev/v4l/by-id'), Path('/dev/v4l/by-path')):
    if directory.exists():
        stable_paths.extend(sorted(directory.glob('*-video-index*')))

inventory = []
seen_device_paths = set()
for stable_path in stable_paths:
    device_path = stable_path.resolve()
    if device_path in seen_device_paths:
        continue
    seen_device_paths.add(device_path)
    cap = cv2.VideoCapture(str(stable_path), cv2.CAP_V4L2)
    try:
        opened = cap.isOpened()
        read_ok, frame = cap.read() if opened else (False, None)
        inventory.append({
            'opencv_id': video_index(device_path),
            'stable_path': str(stable_path),
            'device_path': str(device_path),
            'opened': opened,
            'read_ok': read_ok,
            'frame_shape': tuple(frame.shape) if read_ok else None,
            'backend': cap.getBackendName() if opened else None,
        })
    finally:
        cap.release()

if inventory:
    for entry in inventory:
        status = 'PASS' if entry['read_ok'] else 'FAIL'
        print(f"{status}  dev{entry['opencv_id']}: {entry['stable_path']} -> {entry['device_path']}  frame={entry['frame_shape']}")
else:
    print('FAIL  No stable paths found under /dev/v4l/by-id or /dev/v4l/by-path.')

PASS  dev0: /dev/v4l/by-id/usb-OmniVision_Technologies__Inc._USB_Camera-B4.09.24.1-video-index0 -> /dev/video0  frame=(480, 640, 3)
PASS  dev1: /dev/v4l/by-path/pci-0000:0a:00.3-usb-0:1.2:1.0-video-index0 -> /dev/video1  frame=(480, 640, 3)


At this point you should identify the valid cameras above, and edit the "camera_controller_{machine}.yaml" file to choose the cameras you actually want to use on this machine. 

If the cameras are not accessible, you can verify if something is holding them by typing:
    lsof /dev/video0 /dev/video1
and identifying which processes are holding them (which will likely be a hanging jupyter kernel.)

In [3]:
camera_experiment = 'controllers'
machine = socket.gethostname().split('.', 1)[0]
camera_run = f'camera_controller_{machine}'
exp_camera = Config().get_experiment(camera_experiment, camera_run)
configured_ids = list(exp_camera['active_camera_list'])
preview_size = tuple(exp_camera['saved_image_size'])
working_ids = {entry['opencv_id'] for entry in inventory if entry['read_ok']}
print(f'Configured cameras: {configured_ids}')
print(f'Preview size: {preview_size}')
for camera_id in configured_ids:
    print(f"{'PASS' if camera_id in working_ids else 'FAIL'}  configured dev{camera_id}")

preview_ids = configured_ids  # Change to any passing OpenCV device IDs if needed.

***ExpRun**: Loading pointer config file:
	/home/lboloni/.config/BerryPicker/mainsettings.yaml
***ExpRun**: Loading machine-specific config file:
	/home/lboloni/Insync/lotzi.boloni@gmail.com/Google Drive/LotziStudy/Code/PackageTracking/BerryPicker/settings/settings-tredy2.yaml
***ExpRun**: Using torch device: cuda
***ExpRun**: Experiment default config /home/lboloni/Documents/Hackingwork/_Checkouts/BerryPicker/BerryPicker/src/experiment_configs/controllers/_defaults_controllers.yaml was empty, ok.
***ExpRun**: Configuration for exp/run: controllers/camera_controller_tredy2 successfully loaded
Configured cameras: [0, 1]
Preview size: (512, 512)
PASS  configured dev0
PASS  configured dev1


In [4]:
# Live labelled preview. Cover or move one physical camera at a time to verify its terminal mapping.
captures = {}
try:
    for camera_id in preview_ids:
        cap = cv2.VideoCapture(camera_id, cv2.CAP_V4L2)
        if cap.isOpened():
            cap.set(cv2.CAP_PROP_FRAME_WIDTH, preview_size[0])
            cap.set(cv2.CAP_PROP_FRAME_HEIGHT, preview_size[1])
            captures[camera_id] = cap
        else:
            print(f'FAIL  Unable to open dev{camera_id}')
            cap.release()
    if not captures:
        raise RuntimeError('No configured cameras could be opened.')

    while True:
        tiles = []
        for camera_id, cap in captures.items():
            ok, frame = cap.read()
            if not ok:
                print(f'FAIL  dev{camera_id} stopped returning frames')
                continue
            frame = cv2.resize(frame, preview_size)
            cv2.putText(frame, f'dev{camera_id}', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            tiles.append(frame)
        if not tiles:
            break
        cv2.imshow('Verify_Cameras — q to exit', cv2.hconcat(tiles))
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
finally:
    for cap in captures.values():
        cap.release()
    cv2.destroyAllWindows()
    print('Camera captures released.')

QFontDatabase: Cannot find font directory /home/lboloni/Documents/Hackingwork/_VirtualEnv/venvBerryPicker/venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/lboloni/Documents/Hackingwork/_VirtualEnv/venvBerryPicker/venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/lboloni/Documents/Hackingwork/_VirtualEnv/venvBerryPicker/venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/lboloni/Documents/Hackingwork/_VirtualEnv/venvBerryPicker/venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt

Camera captures released.


A passing setup has a successful one-frame read for every configured device and a live feed whose physical view matches the camera expected at that terminal. Record or update the camera configuration only after completing that visual check.